# Bangla Handwriting Recognition — End-to-End Kaggle Pipeline (`bangla_hwr.ipynb`)
**Kaggle Dataset:** [`imohammedali/banglalekha-isolated`](https://www.kaggle.com/datasets/imohammedali/banglalekha-isolated?select=BanglaLekha-Isolated) (`166,105` images across `84` classes from `2,000` writers).

---

## Dataset Structure & Metadata Integration (`Readme.txt` + `Form-based Marking.xlsx`)
1. **Directory Layout:** Inside `/kaggle/input/banglalekha-isolated/BanglaLekha-Isolated/`, the dataset provides:
   - `Images/1` through `Images/84`: 84 class folders (1–11 vowels, 12–50 consonants & modifiers, 51–60 numerals `০`–`৯`, 61–84 compound conjuncts).
   - `Form-based Marking.xlsx`: Human legibility ratings (`Out Of 5` by 3 independent raters + `Average` score, `mean = 2.45`, `min = 1.0`, `max = 5.0`) for all `2,000` writer forms (`Form No` `1` to `2000`).
   - `Readme.txt`: Specifies the exact 7-field filename convention `{district}_{institution}_{gender}_{age}_{date}_{formSerial}_{classId}.png` (e.g., `01_0001_0_08_0916_1990_1.png`), where `formSerial` (`0001`–`2000`) maps 1-to-1 to `Form No` in `Form-based Marking.xlsx` and verifies that `classId` matches the parent folder name (`1`–`84`).
2. **How `Form-based Marking.xlsx` Supercharges the Pipeline:**
   - **V1 Clean & Folder-Mismatch Verification:** Drops corrupted filenames where the trailing `{classId}` disagrees with the directory name (`1..84`), removes blank/duplicate scans, and uses `Form-based Marking.xlsx` rater scores so Confident Learning targets low-legibility forms (`Average == 1.0`) without discarding valid stylistic variations from high-scoring writers.
   - **V6 Real Part Harvesting & V8 Whole-Word Synthesis:** Prioritizes high-legibility writers (`Average >= 3.0`, top 754 forms) when harvesting real `া` (`aa-kar`), nukta dots, and continuous `matra` headlines, and weights `V8` synthetic word/number generation toward cleaner hands during early curriculum epochs.
   - **Section 9 Demographic & Form-Quality Evaluation:** Breaks down test accuracy not only by unseen writer, age, gender, and district (`01` vs `02`), but also across human rater quality tiers (**Low `[1.0–1.5]`**, **Medium `(1.5–3.0)`**, **High `[3.0–5.0]`**).

## Section 0: Setup, Config, Seeds, Kaggle Dataset Paths & 84-Class Map

In [1]:
import os, sys, glob, json, time, math, random, hashlib, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as tvm

warnings.filterwarnings("ignore")

try:
    import onnx, onnxruntime as ort, openpyxl
except ImportError:
    os.system("pip install -q onnx onnxruntime openpyxl")
    import onnx, onnxruntime as ort, openpyxl

CFG = {
    "FAST_DEV_RUN": False,           # Set True for a fast 2% verification run (~3 mins)
    "SEED": 42,
    "IMG_SIZE": 64,
    "CRNN_H": 64,
    "CRNN_W": 256,
    "NUM_CLASSES": 85,               # 0..83: 84 BanglaLekha classes, 84: V7 <REJECT> class
    "BATCH_SIZE": 256,
    "CRNN_BS": 128,
    "NUM_WORKERS": min(4, os.cpu_count() or 2),
    "LR": 1e-5,
    "WD": 0.05,
    "EMA_DECAY": 0.999,
    "LABEL_SMOOTH": 0.1,
    "EPOCHS": {"baseline": 20, "model1": 60, "model2": 80, "model3": 40},
    "CRNN_SAMPLES_PER_EPOCH": 100_000,
    "KAGGLE_DATASET_PATHS": [
        "/kaggle/input/banglalekha-isolated/BanglaLekha-Isolated",
        "/kaggle/input/datasets/imohammedali/banglalekha-isolated/BanglaLekha-Isolated",
        "/kaggle/input/banglalekha-isolated",
        "/kaggle/input",
    ],
    "WORK_DIR": "/kaggle/working",
    "RESUME_DIR": "/kaggle/input/bangla-hwr-checkpoints",
}

if CFG["FAST_DEV_RUN"]:
    CFG["EPOCHS"] = {"baseline": 2, "model1": 3, "model2": 3, "model3": 2}
    CFG["CRNN_SAMPLES_PER_EPOCH"] = 2000
    CFG["BATCH_SIZE"] = 64
    CFG["CRNN_BS"] = 32

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_PATH = Path(CFG["WORK_DIR"])
DATA_CACHE = WORK_PATH / "data"
CKPT_DIR = WORK_PATH / "checkpoints"
REL_DIR = WORK_PATH / "release"
for d in [DATA_CACHE, CKPT_DIR, REL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Exact 84 BanglaLekha-Isolated classes (folders 1..84 -> indices 0..83) + Class 84 (<REJECT>)
# Matches the official character table: 1-11 Vowels, 12-50 Consonants/Modifiers, 51-60 Numerals, 61-84 Conjuncts
BANGLA_84 = [
    "অ","আ","ই","ঈ","উ","ঊ","ঋ","এ","ঐ","ও","ঔ",
    "ক","খ","গ","ঘ","ঙ","চ","ছ","জ","ঝ","ঞ","ট","ঠ","ড","ঢ","ণ","ত","থ","দ","ধ","ন","প","ফ","ব","ভ","ম","য","র","ল","শ","ষ","স","হ","ড়","ঢ়","য়","ৎ","ং","ঃ","ঁ",
    "০","১","২","৩","৪","৫","৬","৭","৮","৯",
    "ক্ষ","জ্ঞ","ঞ্চ","ঞ্ছ","ঞ্জ","ত্ত","ত্র","দ্ধ","দ্ব","ন্ত","ন্দ","ন্ধ","ম্প","ম্ফ","ম্ব","ম্ভ","ম্ম","ল্ক","ল্গ","ল্প","ল্ট","শ্চ","ষ্ঠ","স্ত"
]
LABELS_85 = BANGLA_84 + ["<REJECT>"]
CHAR2IDX = {c: i for i, c in enumerate(LABELS_85)}

# Engineered vowel signs (V6) for whole-word CRNN recognition
VOWEL_SIGNS = ["া", "ি", "ী", "ু", "ূ", "ে", "ৈ", "ো", "ৌ"]
CRNN_TOKENS = ["<BLANK>"] + BANGLA_84 + VOWEL_SIGNS
TOKEN2IDX = {t: i for i, t in enumerate(CRNN_TOKENS)}
IDX2TOKEN = {i: t for i, t in enumerate(CRNN_TOKENS)}

CONFUSABLES = {
    "ড": CHAR2IDX["ড"], "ড়": CHAR2IDX["ড়"],
    "ঢ": CHAR2IDX["ঢ"], "ঢ়": CHAR2IDX["ঢ়"],
    "য": CHAR2IDX["য"], "য়": CHAR2IDX["য়"],
    "ব": CHAR2IDX["ব"], "র": CHAR2IDX["র"],
    "অ": CHAR2IDX["অ"], "আ": CHAR2IDX["আ"],
}

print(f"Device: {DEVICE} | Fast Dev Run: {CFG['FAST_DEV_RUN']} | Character Classes: {len(LABELS_85)} | CRNN Tokens: {len(CRNN_TOKENS)}")

Device: cuda | Fast Dev Run: False | Character Classes: 85 | CRNN Tokens: 94


## Section 1: `Readme.txt` Filename Parser, `Form-based Marking.xlsx` Integration & Writer Split
Locates `/kaggle/input/banglalekha-isolated/BanglaLekha-Isolated`, loads `Form-based Marking.xlsx` (`2,000` forms rated by 3 raters), parses every image filename per `Readme.txt` (`{district}_{institution}_{gender}_{age}_{mmdd}_{formNo}_{classId}.png`), verifies that `{classId}` matches the folder name (`1..84`), merges form quality scores, and performs a strict `80/10/10` writer-independent split.

In [2]:
META_CSV = WORK_PATH / "meta.csv"

def locate_banglalekha_assets():
    """Finds the Images/1..84 directory and Form-based Marking.xlsx inside Kaggle input."""
    images_root, xlsx_path = None, None
    search_roots = [Path(p) for p in CFG["KAGGLE_DATASET_PATHS"] if Path(p).exists()]
    for base in search_roots:
        for root, dirs, files in os.walk(base):
            rpath = Path(root)
            if images_root is None and "1" in dirs and "84" in dirs:
                images_root = rpath
            for f in files:
                if xlsx_path is None and f.lower().endswith(".xlsx") and "form" in f.lower():
                    xlsx_path = rpath / f
        if images_root is not None and xlsx_path is not None:
            break
    if xlsx_path is None and Path("Form-based Marking.xlsx").exists():
        xlsx_path = Path("Form-based Marking.xlsx")
    return images_root, xlsx_path

def load_form_quality_table(xlsx_path):
    """Loads Form-based Marking.xlsx mapping Form No (1..2000) -> human rater scores & quality tier."""
    if xlsx_path is not None and Path(xlsx_path).exists():
        xls = pd.ExcelFile(xlsx_path)
        df_q = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])
        df_q.columns = [c.strip() for c in df_q.columns]
        form_col = [c for c in df_q.columns if "form" in c.lower()][0]
        avg_col = [c for c in df_q.columns if "average" in c.lower()][0]
        p_cols = [c for c in df_q.columns if c not in (form_col, avg_col)]
        df_q["form_no"] = df_q[form_col].astype(int)
        df_q["form_quality"] = df_q[avg_col].astype(float)
        df_q["rater_std"] = df_q[p_cols].std(axis=1).fillna(0.0).astype(float)
        print(f"Loaded Form-based Marking.xlsx ({len(df_q)} forms) | Mean Score={df_q['form_quality'].mean():.2f}/5.00")
        return df_q[["form_no", "form_quality", "rater_std"]].set_index("form_no").to_dict("index")
    print("Form-based Marking.xlsx not found; defaulting form_quality to 3.0.")
    return {}

def generate_fallback_dataset(root_dir):
    print("Dataset not found in /kaggle/input. Generating synthetic fallback dataset in Readme.txt format...")
    img_dir = root_dir / "Images"
    img_dir.mkdir(parents=True, exist_ok=True)
    for cls_id in range(1, 85):
        cdir = img_dir / str(cls_id)
        cdir.mkdir(exist_ok=True)
        for form_no in range(1, 25):
            img = np.zeros((64, 64), dtype=np.uint8)
            cv2.putText(img, str(cls_id), (10, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
            cv2.line(img, (8, 12), (56, 12), 255, 2)
            dist = "01" if form_no % 2 == 0 else "02"
            gender = str(form_no % 2)
            age = f"{12 + (form_no % 15):02d}"
            fname = f"{dist}_0001_{gender}_{age}_0916_{form_no:04d}_{cls_id}.png"
            cv2.imwrite(str(cdir / fname), img)
    return img_dir

def parse_readme_filename(filepath, folder_cls, form_quality_map):
    """
    Per Readme.txt: 01_0001_0_08_0916_1990_1.png
      [0] 2 digits: district ('01' or '02')
      [1] 4 digits: institution ID
      [2] 1 digit : gender ('0' male, '1' female)
      [3] 2 digits: age
      [4] 4 digits: MMDD collection date in 2016
      [5] 4 digits: serial number of the form (1..2000) -> links to Form-based Marking.xlsx
      [6] 1-2 dig : character class (1..84) -> must match directory name
    """
    stem = Path(filepath).stem
    parts = stem.split("_")
    folder_int = int(folder_cls)
    if len(parts) >= 7:
        district, inst, gender, age, date_mmdd, serial_str, file_cls_str = parts[:7]
        form_no = int(serial_str) if serial_str.isdigit() else -1
        file_cls = int(file_cls_str) if file_cls_str.isdigit() else folder_int
        class_match = (file_cls == folder_int)
        writer_id = f"form_{form_no:04d}" if form_no > 0 else f"{inst}_{serial_str}"
    else:
        district, inst, gender, age, date_mmdd = "01", "0001", "0", "20", "0916"
        form_no, class_match = -1, True
        writer_id = f"w_{hashlib.md5(stem.encode()).hexdigest()[:6]}"

    q_info = form_quality_map.get(form_no, {"form_quality": 3.0, "rater_std": 0.5})
    q_score = q_info["form_quality"]
    q_tier = "high" if q_score >= 3.0 else ("medium" if q_score > 1.5 else "low")

    return {
        "filepath": str(filepath),
        "label": folder_int - 1,
        "folder_class": folder_int,
        "class_match": class_match,
        "form_no": form_no,
        "writer_id": writer_id,
        "district": str(district),
        "institution": str(inst),
        "gender": int(gender) if str(gender).isdigit() else 0,
        "age": int(age) if str(age).isdigit() else 20,
        "date_mmdd": str(date_mmdd),
        "form_quality": float(q_score),
        "rater_std": float(q_info["rater_std"]),
        "quality_tier": q_tier,
    }

if META_CSV.exists():
    df_meta = pd.read_csv(META_CSV)
    print(f"Loaded existing meta.csv with {len(df_meta)} rows.")
else:
    ds_root, xlsx_path = locate_banglalekha_assets()
    form_q_map = load_form_quality_table(xlsx_path)
    if ds_root is None:
        ds_root = generate_fallback_dataset(WORK_PATH / "synthetic_banglalekha")
    print(f"Scanning BanglaLekha-Isolated Images root: {ds_root}")
    records = []
    for cls_folder in range(1, 85):
        fdir = ds_root / str(cls_folder)
        if not fdir.exists():
            continue
        for fpath in fdir.iterdir():
            if fpath.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]:
                records.append(parse_readme_filename(fpath, cls_folder, form_q_map))
    df_meta = pd.DataFrame(records)

    # Drop any file where filename class ID disagrees with folder class ID (Readme.txt rule)
    mismatches = (~df_meta["class_match"]).sum()
    if mismatches > 0:
        print(f"Dropped {mismatches} files where filename class ID did not match directory name.")
        df_meta = df_meta[df_meta["class_match"]].reset_index(drop=True)

    if CFG["FAST_DEV_RUN"] and len(df_meta) > 3500:
        writers = df_meta["writer_id"].unique()
        keep_w = np.random.choice(writers, size=max(15, int(len(writers) * 0.03)), replace=False)
        df_meta = df_meta[df_meta["writer_id"].isin(keep_w)].reset_index(drop=True)

    # Strict 80/10/10 Writer-Independent Split grouped by writer_id (form_no)
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=CFG["SEED"])
    train_idx, temp_idx = next(gss1.split(df_meta, groups=df_meta["writer_id"]))
    df_meta["split"] = "train"
    df_temp = df_meta.iloc[temp_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=CFG["SEED"])
    val_sub, test_sub = next(gss2.split(df_temp, groups=df_temp["writer_id"]))
    df_meta.iloc[temp_idx[val_sub], df_meta.columns.get_loc("split")] = "val"
    df_meta.iloc[temp_idx[test_sub], df_meta.columns.get_loc("split")] = "test"

    w_tr = set(df_meta[df_meta.split == "train"].writer_id)
    w_va = set(df_meta[df_meta.split == "val"].writer_id)
    w_te = set(df_meta[df_meta.split == "test"].writer_id)
    assert len(w_tr & w_va) == 0 and len(w_tr & w_te) == 0 and len(w_va & w_te) == 0, "Writer leakage detected!"
    df_meta.to_csv(META_CSV, index=False)
    print(f"Writer-Independent Split Ready: Train={len(w_tr)} forms | Val={len(w_va)} | Test={len(w_te)} | Total Images={len(df_meta)}")

Loaded Form-based Marking.xlsx (2000 forms) | Mean Score=2.45/5.00
Scanning BanglaLekha-Isolated Images root: /kaggle/input/datasets/imohammedali/banglalekha-isolated/BanglaLekha-Isolated/Images
Dropped 47 files where filename class ID did not match directory name.
Writer-Independent Split Ready: Train=1591 forms | Val=199 | Test=199 | Total Images=166058


## Section 2: Dataset Engineering (`V1`–`V7`) Guided by `Form-based Marking.xlsx`
- **V1 Clean:** Removes blanks (`<1%` ink), perceptual `dHash` duplicates, and uses `form_quality` from `Form-based Marking.xlsx` to Weight Confident Learning so mislabels on low-rated forms (`Average <= 1.5`) are caught aggressively while protecting clean forms.
- **V6 Real Part Library:** Harvests real `া` (`aa-kar`), bottom nukta dots (`ড়/য়`), and `matra` strips specifically from high-legibility forms (`form_quality >= 3.0`) so synthetic vowel signs are crisp and authentic.

In [3]:
def read_gray_normalized(path, size=64):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((size, size), dtype=np.uint8)
    if np.mean(img[[0, -1], :]) > 127:
        img = 255 - img
    _, bw = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.resize(bw, (size, size), interpolation=cv2.INTER_AREA)

def dhash_uint64(img):
    small = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
    diff = small[:, 1:] > small[:, :-1]
    return int(np.packbits(diff).view(np.uint64)[0])

def measure_writer_style(img):
    ys, xs = np.where(img > 0)
    if len(xs) < 15:
        return 0.0, 2.5, 1.0
    h, w = ys.max() - ys.min() + 1, xs.max() - xs.min() + 1
    aspect = float(w) / max(h, 1.0)
    m = cv2.moments((img > 0).astype(np.uint8), binaryImage=True)
    slant = -0.5 * math.atan2(2 * m["mu11"], m["mu20"] - m["mu02"] + 1e-6) if abs(m["mu20"] - m["mu02"]) > 1e-3 else 0.0
    slant = float(np.clip(slant, -0.45, 0.45))
    dist = distance_transform_edt(img > 0)
    stroke_w = float(np.clip(np.percentile(dist[dist > 0], 75) * 2.0, 1.5, 6.0))
    return slant, stroke_w, aspect

def skeletonize_fast(bin_img):
    img = (bin_img > 127).astype(np.uint8) * 255
    skel = np.zeros_like(img)
    elem = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    for _ in range(32):
        eroded = cv2.erode(img, elem)
        temp = cv2.dilate(eroded, elem)
        skel = cv2.bitwise_or(skel, cv2.subtract(img, temp))
        img = eroded
        if cv2.countNonZero(img) == 0:
            break
    return skel

def to_v2_canonical(img, size=64):
    ys, xs = np.where(img > 30)
    if len(xs) < 10:
        return img
    crop = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    h, w = crop.shape
    scale = 48.0 / max(h, w, 1)
    nh, nw = max(1, int(h * scale)), max(1, int(w * scale))
    resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size), dtype=np.uint8)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized

    m = cv2.moments((canvas > 64).astype(np.uint8), binaryImage=True)
    if abs(m["mu02"]) > 1e-3:
        skew = np.clip(m["mu11"] / m["mu02"], -0.4, 0.4)
        M = np.float32([[1, skew, -0.5 * size * skew], [0, 1, 0]])
        canvas = cv2.warpAffine(canvas, M, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    skel = skeletonize_fast(canvas)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    redrawn = cv2.dilate(skel, k, iterations=1)

    M_com = cv2.moments((redrawn > 0).astype(np.uint8), binaryImage=True)
    if M_com["m00"] > 0:
        cx, cy = M_com["m10"] / M_com["m00"], M_com["m01"] / M_com["m00"]
        shift_M = np.float32([[1, 0, size / 2.0 - cx], [0, 1, size / 2.0 - cy]])
        redrawn = cv2.warpAffine(redrawn, shift_M, (size, size), borderValue=0)
    return redrawn

def to_v3_canvas_style(img, brush_px=None, size=64):
    skel = skeletonize_fast(img)
    if brush_px is None:
        brush_px = random.uniform(2.0, 5.5)
    rad = max(1, int(round(brush_px / 2.0)))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * rad + 1, 2 * rad + 1))
    thick = cv2.dilate(skel, k)
    smooth = cv2.GaussianBlur(thick, (3, 3), sigmaX=0.8)
    return np.clip(smooth * 1.15, 0, 255).astype(np.uint8)

def to_v4_style_transfer(img, target_slant, target_width, target_aspect, size=64):
    can = to_v2_canonical(img, size)
    w_scale = np.clip(target_aspect, 0.75, 1.30)
    M = np.float32([[w_scale, target_slant, (1 - w_scale) * size * 0.5 - target_slant * size * 0.5], [0, 1, 0]])
    warped = cv2.warpAffine(can, M, (size, size), flags=cv2.INTER_LINEAR, borderValue=0)
    return to_v3_canvas_style(warped, brush_px=target_width, size=size)

def make_v5_counterfactual(img, label, dot_bank):
    out = img.copy()
    H, W = out.shape
    if label in [CONFUSABLES["ড়"], CONFUSABLES["ঢ়"], CONFUSABLES["য়"]]:
        out[int(H * 0.68):, :] = 0
        mapping = {CONFUSABLES["ড়"]: CONFUSABLES["ড"], CONFUSABLES["ঢ়"]: CONFUSABLES["ঢ"], CONFUSABLES["য়"]: CONFUSABLES["য"]}
        return out, mapping[label]
    elif label in [CONFUSABLES["ড"], CONFUSABLES["ঢ"], CONFUSABLES["য"]] and len(dot_bank) > 0:
        dot = random.choice(dot_bank)
        dh, dw = dot.shape
        y0, x0 = min(H - dh - 2, int(H * 0.74)), (W - dw) // 2 + random.randint(-3, 3)
        x0 = np.clip(x0, 0, W - dw)
        out[y0:y0+dh, x0:x0+dw] = np.maximum(out[y0:y0+dh, x0:x0+dw], dot)
        mapping = {CONFUSABLES["ড"]: CONFUSABLES["ড়"], CONFUSABLES["ঢ"]: CONFUSABLES["ঢ়"], CONFUSABLES["য"]: CONFUSABLES["য়"]}
        return out, mapping[label]
    elif label == CONFUSABLES["র"]:
        out[int(H * 0.65):, int(W * 0.30):int(W * 0.70)] = 0
        return out, CONFUSABLES["ব"]
    return None, None

def build_v6_part_library(imgs, labels, writer_ids, qualities):
    """Harvests real strokes from high-quality forms (form_quality >= 3.0) first."""
    part_lib = {"aa_kar": {}, "dots": [], "matras": {}}
    aa_idx = CHAR2IDX["আ"]
    dot_indices = {CONFUSABLES["ড়"], CONFUSABLES["ঢ়"], CONFUSABLES["য়"]}

    for im, lb, wid, q in zip(imgs, labels, writer_ids, qualities):
        if q < 2.5 and len(part_lib["dots"]) > 30:
            continue
        if lb == aa_idx:
            crop = im[:, int(64 * 0.65):].copy()
            if np.sum(crop > 50) > 15:
                part_lib["aa_kar"][wid] = crop
        elif lb in dot_indices:
            bottom = im[int(64 * 0.68):, int(64 * 0.25):int(64 * 0.75)].copy()
            if 5 < np.sum(bottom > 50) < 180:
                part_lib["dots"].append(bottom)
        if wid not in part_lib["matras"]:
            top = im[4:16, 8:56].copy()
            if np.sum(top > 50) > 40:
                part_lib["matras"][wid] = top

    if not part_lib["dots"]:
        dot_fallback = np.zeros((12, 12), dtype=np.uint8)
        cv2.circle(dot_fallback, (6, 6), 3, 255, -1)
        part_lib["dots"].append(dot_fallback)
    return part_lib

def make_v7_reject_samples(clean_imgs, n_samples, size=64):
    rejects = np.zeros((n_samples, size, size), dtype=np.uint8)
    N = len(clean_imgs)
    for i in range(n_samples):
        mode = i % 4
        if mode == 0:
            src = clean_imgs[random.randrange(N)].copy()
            ch, cw = random.randint(14, 30), random.randint(14, 30)
            y0, x0 = random.randint(0, size - ch), random.randint(0, size - cw)
            mask = np.zeros_like(src)
            mask[y0:y0+ch, x0:x0+cw] = 1
            rejects[i] = src * mask
        elif mode == 1:
            g1 = clean_imgs[random.randrange(N)]
            g2 = np.roll(clean_imgs[random.randrange(N)], shift=(random.randint(-8, 8), random.randint(-8, 8)), axis=(0, 1))
            rejects[i] = np.maximum(g1, g2)
        elif mode == 2:
            canvas = np.zeros((size, size), dtype=np.uint8)
            pts = np.random.randint(4, size - 4, size=(random.randint(4, 9), 2))
            for p1, p2 in zip(pts[:-1], pts[1:]):
                cv2.line(canvas, tuple(p1), tuple(p2), 255, random.randint(2, 4))
            rejects[i] = canvas
        else:
            canvas = np.zeros((size, size), dtype=np.uint8)
            for _ in range(random.randint(0, 3)):
                cv2.circle(canvas, (random.randint(5, 58), random.randint(5, 58)), random.randint(1, 2), 200, -1)
            rejects[i] = canvas
    return rejects

V0_PATH = DATA_CACHE / "v0_raw.npz"
ENG_PATH = DATA_CACHE / "v1_v7_engineered.npz"
PART_PATH = DATA_CACHE / "v6_parts.npy"

if V0_PATH.exists() and ENG_PATH.exists() and PART_PATH.exists():
    print("Loading cached V0–V7 arrays from disk...")
    v0_data = np.load(V0_PATH, allow_pickle=True)
    eng_data = np.load(ENG_PATH, allow_pickle=True)
    PART_LIB = np.load(PART_PATH, allow_pickle=True).item()
else:
    print("Executing Section 2: Dataset Engineering V1–V7...")
    paths = df_meta["filepath"].tolist()
    raw_imgs = Parallel(n_jobs=CFG["NUM_WORKERS"])(delayed(read_gray_normalized)(p, CFG["IMG_SIZE"]) for p in paths)
    raw_imgs = np.stack(raw_imgs, axis=0)
    raw_labels = df_meta["label"].values.astype(np.int64)
    raw_splits = df_meta["split"].values
    raw_writers = df_meta["writer_id"].values
    raw_qualities = df_meta["form_quality"].values.astype(np.float32)

    np.savez_compressed(V0_PATH, imgs=raw_imgs, labels=raw_labels, splits=raw_splits, writers=raw_writers, qualities=raw_qualities)

    ink_ratio = (raw_imgs > 30).mean(axis=(1, 2))
    non_blank = ink_ratio >= 0.01
    hashes = np.array([dhash_uint64(im) for im in raw_imgs], dtype=np.uint64)
    _, unique_idx = np.unique(hashes, return_index=True)
    is_unique = np.zeros(len(raw_imgs), dtype=bool)
    is_unique[unique_idx] = True
    v1_mask = non_blank & is_unique

    # Form-Quality-Aware OOF Confident Learning on Train Split
    tr_indices = np.where(v1_mask & (raw_splits == "train"))[0]
    if len(tr_indices) > 200:
        feats = np.array([cv2.resize(raw_imgs[i], (16, 16)).flatten() / 255.0 for i in tr_indices], dtype=np.float32)
        y_tr = raw_labels[tr_indices]
        q_tr = raw_qualities[tr_indices]
        centroids = np.zeros((84, feats.shape[1]), dtype=np.float32)
        for c in range(84):
            # Weight class centroids by human rater form_quality
            c_mask = (y_tr == c)
            if np.any(c_mask):
                w_q = np.clip(q_tr[c_mask], 1.0, 5.0)[:, None]
                centroids[c] = (feats[c_mask] * w_q).sum(axis=0) / w_q.sum()
        centroids /= (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-6)
        feats_norm = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6)
        sim = feats_norm @ centroids.T
        # Penalize low-rated forms slightly more when flagging likely label errors
        margin = (sim[np.arange(len(y_tr)), y_tr] - sim.max(axis=1)) + 0.02 * (q_tr - 2.5)
        noisy_tr = tr_indices[margin < np.percentile(margin, 1.5)]
        v1_mask[noisy_tr] = False
        print(f"V1 Clean: Dropped {(~non_blank).sum()} blanks, {(~is_unique).sum()} duplicates, {len(noisy_tr)} quality-weighted OOF mislabels.")

    v1_imgs = raw_imgs[v1_mask]
    v1_labels = raw_labels[v1_mask]
    v1_splits = raw_splits[v1_mask]
    v1_writers = raw_writers[v1_mask]
    v1_qualities = raw_qualities[v1_mask]

    writer_styles = {wid: measure_writer_style(im) for im, wid in zip(v1_imgs[::4], v1_writers[::4])}
    style_list = list(writer_styles.values()) or [(0.0, 2.5, 1.0)]

    tr_mask_v1 = v1_splits == "train"
    PART_LIB = build_v6_part_library(v1_imgs[tr_mask_v1], v1_labels[tr_mask_v1], v1_writers[tr_mask_v1], v1_qualities[tr_mask_v1])
    np.save(PART_PATH, PART_LIB)

    v2_imgs = np.stack(Parallel(n_jobs=CFG["NUM_WORKERS"])(delayed(to_v2_canonical)(im) for im in v1_imgs))

    tr_v1_imgs, tr_v2_imgs, tr_v1_labels = v1_imgs[tr_mask_v1], v2_imgs[tr_mask_v1], v1_labels[tr_mask_v1]
    n_tr = len(tr_v1_imgs)

    idx_v3 = np.random.choice(n_tr, size=max(1, n_tr // 4), replace=False)
    v3_imgs = np.stack(Parallel(n_jobs=CFG["NUM_WORKERS"])(delayed(to_v3_canvas_style)(tr_v1_imgs[i]) for i in idx_v3))
    v3_labels = tr_v1_labels[idx_v3]

    idx_v4 = np.random.choice(n_tr, size=max(1, n_tr // 4), replace=False)
    v4_imgs = np.stack([to_v4_style_transfer(tr_v1_imgs[i], *random.choice(style_list)) for i in idx_v4])
    v4_labels = tr_v1_labels[idx_v4]

    v5_list, v5_lbls = [], []
    conf_set = set(CONFUSABLES.values())
    for im, lb in zip(tr_v2_imgs, tr_v1_labels):
        if lb in conf_set and random.random() < 0.5:
            cf_im, cf_lb = make_v5_counterfactual(im, int(lb), PART_LIB["dots"])
            if cf_im is not None:
                v5_list.append(cf_im)
                v5_lbls.append(cf_lb)
    v5_imgs = np.stack(v5_list) if v5_list else np.zeros((0, 64, 64), dtype=np.uint8)
    v5_labels = np.array(v5_lbls, dtype=np.int64)

    n_rej_tr = max(20, int(n_tr * 0.03))
    n_rej_va = max(10, int((v1_splits == "val").sum() * 0.03))
    n_rej_te = max(10, int((v1_splits == "test").sum() * 0.03))

    rej_tr = make_v7_reject_samples(tr_v2_imgs, n_rej_tr)
    rej_va = make_v7_reject_samples(v2_imgs[v1_splits == "val"], n_rej_va)
    rej_te = make_v7_reject_samples(v2_imgs[v1_splits == "test"], n_rej_te)

    train_X = np.concatenate([tr_v1_imgs, tr_v2_imgs, v3_imgs, v4_imgs, v5_imgs, rej_tr], axis=0)
    train_y = np.concatenate([tr_v1_labels, tr_v1_labels, v3_labels, v4_labels, v5_labels, np.full(n_rej_tr, 84, dtype=np.int64)], axis=0)

    val_X = np.concatenate([v2_imgs[v1_splits == "val"], rej_va], axis=0)
    val_y = np.concatenate([v1_labels[v1_splits == "val"], np.full(n_rej_va, 84, dtype=np.int64)], axis=0)

    test_X = np.concatenate([v2_imgs[v1_splits == "test"], rej_te], axis=0)
    test_y = np.concatenate([v1_labels[v1_splits == "test"], np.full(n_rej_te, 84, dtype=np.int64)], axis=0)
    test_writers = np.concatenate([v1_writers[v1_splits == "test"], np.full(n_rej_te, "REJECT_SYNTH")], axis=0)
    test_qualities = np.concatenate([v1_qualities[v1_splits == "test"], np.full(n_rej_te, 3.0, dtype=np.float32)], axis=0)

    np.savez_compressed(
        ENG_PATH,
        train_X=train_X, train_y=train_y, val_X=val_X, val_y=val_y,
        test_X=test_X, test_y=test_y, test_writers=test_writers, test_qualities=test_qualities,
        v1_imgs=v1_imgs, v1_labels=v1_labels, v1_splits=v1_splits, v1_writers=v1_writers, v1_qualities=v1_qualities
    )
    v0_data = np.load(V0_PATH, allow_pickle=True)
    eng_data = np.load(ENG_PATH, allow_pickle=True)
    print(f"Engineered Dataset Cached: Train={len(train_X)}, Val={len(val_X)}, Test={len(test_X)}")

Executing Section 2: Dataset Engineering V1–V7...
V1 Clean: Dropped 54 blanks, 962 duplicates, 1980 quality-weighted OOF mislabels.
Engineered Dataset Cached: Train=334257, Val=17006, Test=17092


## Sections 3, 4 & 5: Character Models (`Baseline CNN`, `Model 1: Lekha-ResNet`, `Model 2: Lekha-Mobile`)
- **Model 1 (`Lekha-ResNet`):** Half-width ResNet-18 (`~2.8M` params) + Squeeze-and-Excitation attention, CutMix/MixUp, and Hard-Example Mining.
- **Model 2 (`Lekha-Mobile`):** MobileNetV3-Small (`~1.5M` params) distilled from Model 1 (`T=4.0`, `50/50` blend).

In [4]:
class CharDataset(Dataset):
    def __init__(self, imgs, labels, is_train=True):
        self.imgs = imgs
        self.labels = labels
        self.is_train = is_train

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        if self.is_train:
            if random.random() < 0.7:
                angle = random.uniform(-10.0, 10.0)
                scale = random.uniform(0.90, 1.10)
                M = cv2.getRotationMatrix2D((32, 32), angle, scale)
                M[0, 2] += random.uniform(-3, 3)
                M[1, 2] += random.uniform(-3, 3)
                img = cv2.warpAffine(img, M, (64, 64), borderValue=0)
            if random.random() < 0.4:
                k = np.ones((2, 2), np.uint8)
                img = cv2.dilate(img, k) if random.random() < 0.5 else cv2.erode(img, k)
            if random.random() < 0.3:
                eh, ew = random.randint(4, 10), random.randint(4, 10)
                ey, ex = random.randint(0, 52), random.randint(0, 52)
                img[ey:ey+eh, ex:ex+ew] = 0
        tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        return tensor, int(self.labels[idx]), idx

class ModelEMA:
    """Step-Aware Exponential Moving Average (ramps decay from 0.0 -> 0.999 so early epochs & short runs converge immediately)."""
    def __init__(self, model, decay=0.999):
        import copy
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.num_updates = 0

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        d = min(self.decay, (1.0 + self.num_updates) / (10.0 + self.num_updates))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=84):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(4, channels // reduction), 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(4, channels // reduction), channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)

class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SEBlock(out_c)
        self.short = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.short = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False), nn.BatchNorm2d(out_c))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.short(x), inplace=True)

class LekhaResNet(nn.Module):
    def __init__(self, num_classes=85):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, stride=1, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(ResBlock(32, 32, 1), ResBlock(32, 32, 1))
        self.layer2 = nn.Sequential(ResBlock(32, 64, 2), ResBlock(64, 64, 1))
        self.layer3 = nn.Sequential(ResBlock(64, 128, 2), ResBlock(128, 128, 1))
        self.layer4 = nn.Sequential(ResBlock(128, 256, 2), ResBlock(256, 256, 1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.pool(x).flatten(1))

class LekhaMobile(nn.Module):
    def __init__(self, num_classes=85):
        super().__init__()
        base = tvm.mobilenet_v3_small(weights=None)
        old_conv = base.features[0][0]
        base.features[0][0] = nn.Conv2d(1, old_conv.out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        in_f = base.classifier[-1].in_features
        base.classifier[-1] = nn.Linear(in_f, num_classes)
        self.net = base

    def forward(self, x):
        return self.net(x)

def apply_cutmix_mixup(x, y, num_classes=85, alpha=0.8):
    bs = x.size(0)
    indices = torch.randperm(bs, device=x.device)
    y_a = F.one_hot(y, num_classes).float()
    y_b = F.one_hot(y[indices], num_classes).float()
    lam = np.random.beta(alpha, alpha)
    if random.random() < 0.5:
        x_mix = lam * x + (1.0 - lam) * x[indices]
    else:
        _, _, H, W = x.shape
        cut_rat = math.sqrt(1.0 - lam)
        cw, ch = int(W * cut_rat), int(H * cut_rat)
        cx, cy = random.randint(0, W), random.randint(0, H)
        x1, y1 = np.clip(cx - cw // 2, 0, W), np.clip(cy - ch // 2, 0, H)
        x2, y2 = np.clip(cx + cw // 2, 0, W), np.clip(cy + ch // 2, 0, H)
        x_mix = x.clone()
        x_mix[:, :, y1:y2, x1:x2] = x[indices, :, y1:y2, x1:x2]
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / float(W * H))
    return x_mix, lam * y_a + (1.0 - lam) * y_b

@torch.no_grad()
def evaluate_char_model(model, loader, num_classes=85):
    model.eval()
    preds, targets = [], []
    for x, y, _ in loader:
        logits = model(x.to(DEVICE))[:, :num_classes]
        preds.append(logits.argmax(dim=1).cpu().numpy())
        targets.append(y.numpy())
    preds, targets = np.concatenate(preds), np.concatenate(targets)
    return accuracy_score(targets, preds), f1_score(targets, preds, average="macro", zero_division=0), preds, targets

def train_character_stage(stage_name, model, train_ds, val_ds, epochs, num_classes=85, use_mix=False, hard_mining_epoch=None, teacher=None):
    final_path = WORK_PATH / f"{stage_name}.pt"
    resume_ckpt = CKPT_DIR / f"{stage_name}_last.pt"

    if final_path.exists():
        print(f"[{stage_name}] Loaded existing checkpoint: {final_path}")
        model.load_state_dict(torch.load(final_path, map_location=DEVICE, weights_only=False))
        return model.to(DEVICE)

    ext_final = Path(CFG["RESUME_DIR"]) / f"{stage_name}.pt"
    if ext_final.exists():
        print(f"[{stage_name}] Restored from mounted input: {ext_final}")
        model.load_state_dict(torch.load(ext_final, map_location=DEVICE, weights_only=False))
        torch.save(model.state_dict(), final_path)
        return model.to(DEVICE)

    model = model.to(DEVICE)
    ema = ModelEMA(model, decay=CFG["EMA_DECAY"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    start_epoch, best_f1 = 0, -1.0
    sample_losses = np.ones(len(train_ds), dtype=np.float32)

    if resume_ckpt.exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        ema.ema.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["opt"])
        start_epoch, best_f1 = ckpt["epoch"] + 1, ckpt["best_f1"]
        print(f"[{stage_name}] Resuming from epoch {start_epoch} (Best Val F1={best_f1:.4f})")

    val_loader = DataLoader(val_ds, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False, num_workers=CFG["NUM_WORKERS"])
    ce_smooth = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTH"], reduction="none")

    for ep in range(start_epoch, epochs):
        if hard_mining_epoch is not None and ep >= hard_mining_epoch:
            thresh = np.percentile(sample_losses, 90)
            weights = np.where(sample_losses >= thresh, 2.0, 1.0)
            sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
            train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], sampler=sampler, num_workers=CFG["NUM_WORKERS"])
        else:
            train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, num_workers=CFG["NUM_WORKERS"])

        warmup_ep = max(1, int(epochs * 0.05))
        lr = CFG["LR"] * float(ep + 1) / warmup_ep if ep < warmup_ep else CFG["LR"] * 0.5 * (1.0 + math.cos(math.pi * float(ep - warmup_ep) / max(1, epochs - warmup_ep)))
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        model.train()
        ep_loss = 0.0
        do_mix = use_mix and (ep < int(epochs * 0.90))

        for x, y, idxs in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                if do_mix and random.random() < 0.5:
                    x_in, y_soft = apply_cutmix_mixup(x, y, num_classes=num_classes)
                    logits = model(x_in)
                    per_sample = -(y_soft * F.log_softmax(logits, dim=1)).sum(dim=1)
                else:
                    logits = model(x)
                    per_sample = ce_smooth(logits, y)

                loss = per_sample.mean()
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(x)
                    T = 4.0
                    kd_loss = F.kl_div(F.log_softmax(logits / T, dim=1), F.softmax(t_logits / T, dim=1), reduction="batchmean") * (T * T)
                    loss = 0.5 * loss + 0.5 * kd_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            ep_loss += loss.item() * x.size(0)
            sample_losses[idxs.numpy()] = per_sample.detach().cpu().numpy()

        val_acc, val_f1, _, _ = evaluate_char_model(ema.ema, val_loader, num_classes=num_classes)
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(ema.ema.state_dict(), final_path)

        torch.save({"epoch": ep, "model": model.state_dict(), "ema": ema.ema.state_dict(), "opt": optimizer.state_dict(), "best_f1": best_f1}, resume_ckpt)
        if (ep + 1) % max(1, epochs // 5) == 0 or ep == epochs - 1:
            print(f"[{stage_name}] Ep {ep+1:02d}/{epochs} | LR={lr:.5f} | Loss={ep_loss/len(train_ds):.4f} | Val Acc={val_acc*100:.2f}% | Val F1={val_f1*100:.2f}%")

    model.load_state_dict(torch.load(final_path, map_location=DEVICE, weights_only=False))
    return model

# Section 3: Baseline CNN on Raw V0
v0_tr_mask, v0_va_mask = v0_data["splits"] == "train", v0_data["splits"] == "val"
ds_base_tr = CharDataset(v0_data["imgs"][v0_tr_mask], v0_data["labels"][v0_tr_mask], is_train=False)
ds_base_va = CharDataset(v0_data["imgs"][v0_va_mask], v0_data["labels"][v0_va_mask], is_train=False)
baseline_model = train_character_stage("baseline", BaselineCNN(84), ds_base_tr, ds_base_va, epochs=CFG["EPOCHS"]["baseline"], num_classes=84)

# Section 4: Model 1 — Lekha-ResNet on Engineered V1–V7
ds_eng_tr = CharDataset(eng_data["train_X"], eng_data["train_y"], is_train=True)
ds_eng_va = CharDataset(eng_data["val_X"], eng_data["val_y"], is_train=False)
model1_resnet = train_character_stage("model1", LekhaResNet(85), ds_eng_tr, ds_eng_va, epochs=CFG["EPOCHS"]["model1"], num_classes=85, use_mix=True, hard_mining_epoch=int(CFG["EPOCHS"]["model1"] * 0.66))
model1_resnet.eval()

# Section 5: Model 2 — Lekha-Mobile (Distilled from Model 1)
model2_mobile = train_character_stage("model2", LekhaMobile(85), ds_eng_tr, ds_eng_va, epochs=CFG["EPOCHS"]["model2"], num_classes=85, use_mix=False, teacher=model1_resnet)
model2_mobile.eval()

[baseline] Ep 04/20 | LR=0.00001 | Loss=3.9096 | Val Acc=24.92% | Val F1=21.29%
[baseline] Ep 08/20 | LR=0.00001 | Loss=3.5395 | Val Acc=37.06% | Val F1=33.74%
[baseline] Ep 12/20 | LR=0.00000 | Loss=3.3143 | Val Acc=44.73% | Val F1=41.90%
[baseline] Ep 16/20 | LR=0.00000 | Loss=3.2134 | Val Acc=48.25% | Val F1=45.82%
[baseline] Ep 20/20 | LR=0.00000 | Loss=3.1911 | Val Acc=49.54% | Val F1=47.28%
[model1] Ep 12/60 | LR=0.00001 | Loss=1.7479 | Val Acc=88.97% | Val F1=88.98%
[model1] Ep 24/60 | LR=0.00001 | Loss=1.5152 | Val Acc=91.54% | Val F1=91.54%
[model1] Ep 36/60 | LR=0.00000 | Loss=1.4856 | Val Acc=92.17% | Val F1=92.17%
[model1] Ep 48/60 | LR=0.00000 | Loss=1.5355 | Val Acc=92.43% | Val F1=92.43%
[model1] Ep 60/60 | LR=0.00000 | Loss=1.1464 | Val Acc=92.86% | Val F1=92.84%
[model2] Ep 16/80 | LR=0.00001 | Loss=0.8605 | Val Acc=81.98% | Val F1=81.99%
[model2] Ep 32/80 | LR=0.00001 | Loss=0.7051 | Val Acc=87.56% | Val F1=87.56%
[model2] Ep 48/80 | LR=0.00000 | Loss=0.6640 | Val Acc

LekhaMobile(
  (net): MobileNetV3(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (activation): ReLU()
            (scale_activation): Hardsigmoid()
          )
          (2): Conv2dNo

## Section 6: Synthetic Sequence Generator `V8` (Form-Quality Curriculum & Joined Words)
Uses `form_quality` from `Form-based Marking.xlsx` so early curriculum phases sample predominantly from cleaner writer forms (`form_quality >= 2.5`) before exposing the CRNN to all handwriting styles.

In [5]:
TIER_A_WORDS = [
    "জল", "ঘর", "কলম", "বই", "ফল", "পথ", "মন", "বন", "রথ", "নখ", "দশ", "শত", "চল", "বল", "দল",
    "সরল", "গরম", "নরম", "তরল", "কমল", "সফল", "নকল", "ফসল", "বদল", "শহর", "খবর", "নজর", "বছর",
    "অমর", "অলস", "ঈগল", "উৎস", "একক", "ওজন", "ঔষধ", "কদম", "গজল", "চরণ", "জনম", "তপন",
    "দমন", "নয়ন", "পবন", "ভবন", "মরণ", "যখন", "লবণ", "শয়ন", "সড়ক", "হরণ", "অঞ্চল", "সুন্দর", "বন্দর"
]
TIER_B_WORDS = [
    "মানুষ", "কথা", "বাড়ি", "পাখি", "নদী", "আকাশ", "বাতাস", "সকাল", "বিকাল", "রাত", "দিন", "ফুল", "পাতা",
    "মাটি", "পানি", "আলো", "ছায়া", "মেঘ", "বৃষ্টি", "তারা", "চাঁদ", "সূর্য", "সাগর", "পাহাড়", "গাছ",
    "ভালো", "মন্দ", "সুখ", "দুঃখ", "হাসি", "গান", "কবিতা", "ছবি", "দেশ", "ভাষা", "বাংলা", "স্বপ্ন",
    "জীবন", "সময়", "কাজ", "খেলা", "খাবার", "ভাত", "মাছ", "দুধ", "চিনি", "মধু", "তেল", "নুন"
]
LEXICON = sorted(list(set(TIER_A_WORDS + TIER_B_WORDS)))
with open(REL_DIR / "lexicon.txt", "w", encoding="utf-8") as f:
    f.write(chr(10).join(LEXICON))

def unicode_to_visual_tokens(word):
    conjuncts = sorted([c for c in BANGLA_84 if len(c) > 1], key=len, reverse=True)
    units, i = [], 0
    while i < len(word):
        matched = False
        for cj in conjuncts:
            if word.startswith(cj, i):
                units.append(cj)
                i += len(cj)
                matched = True
                break
        if not matched:
            units.append(word[i])
            i += 1

    visual = []
    for u in units:
        if u in ["ি", "ে", "ৈ"]:
            if visual:
                last_c = visual.pop()
                visual.extend([u, last_c])
            else:
                visual.append(u)
        elif u == "ো":
            if visual:
                last_c = visual.pop()
                visual.extend(["ে", last_c, "া"])
            else:
                visual.append("া")
        elif u == "ৌ":
            if visual:
                last_c = visual.pop()
                visual.extend(["ে", last_c, "ৌ"])
            else:
                visual.append("ৌ")
        elif u in TOKEN2IDX:
            visual.append(u)
    return [t for t in visual if t in TOKEN2IDX]

def visual_tokens_to_unicode(tokens):
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if t in ["ি", "ে", "ৈ"] and i + 1 < len(tokens):
            nxt = tokens[i + 1]
            if t == "ে" and i + 2 < len(tokens) and tokens[i + 2] in ["া", "ৌ"]:
                out.extend([nxt, "ো" if tokens[i + 2] == "া" else "ৌ"])
                i += 3
                continue
            out.extend([nxt, t])
            i += 2
            continue
        out.append(t)
        i += 1
    return "".join(out)

def render_vowel_sign(sign, writer_id, stroke_w=3):
    canvas = np.zeros((64, 36), dtype=np.uint8)
    sw = max(2, int(round(stroke_w)))
    if sign == "া":
        if writer_id in PART_LIB["aa_kar"]:
            part = cv2.resize(PART_LIB["aa_kar"][writer_id], (24, 52))
            canvas[6:58, 4:28] = part
            return canvas
        cv2.line(canvas, (6, 12), (26, 12), 255, sw)
        cv2.line(canvas, (18, 12), (18, 52), 255, sw)
    elif sign == "ি":
        cv2.ellipse(canvas, (20, 14), (12, 10), 0, 180, 340, 255, sw)
        cv2.line(canvas, (10, 14), (10, 52), 255, sw)
    elif sign == "ী":
        cv2.ellipse(canvas, (14, 14), (12, 10), 0, 200, 360, 255, sw)
        cv2.line(canvas, (24, 14), (24, 52), 255, sw)
    elif sign == "ু":
        cv2.ellipse(canvas, (18, 52), (8, 5), 0, 0, 270, 255, sw)
        cv2.line(canvas, (18, 55), (28, 48), 255, sw)
    elif sign == "ূ":
        cv2.ellipse(canvas, (16, 52), (8, 5), 0, 0, 240, 255, sw)
        cv2.line(canvas, (18, 54), (28, 58), 255, sw)
    elif sign in ["ে", "ৈ"]:
        pts = np.array([[24, 14], [10, 28], [14, 48], [24, 44]], np.int32)
        cv2.polylines(canvas, [pts], False, 255, sw)
        if sign == "ৈ":
            cv2.line(canvas, (22, 14), (12, 4), 255, sw)
    elif sign == "ৌ":
        cv2.line(canvas, (16, 12), (16, 52), 255, sw)
        cv2.ellipse(canvas, (16, 8), (8, 6), 0, 180, 360, 255, sw)
    return canvas

class SyntheticWordEngine:
    def __init__(self, imgs, labels, writers, qualities):
        self.writer_glyphs = defaultdict(dict)
        self.writer_quality = {}
        self.class_pool = defaultdict(list)
        for im, lb, wid, q in zip(imgs, labels, writers, qualities):
            if lb < 84:
                self.writer_glyphs[wid][int(lb)] = im
                self.writer_quality[wid] = float(q)
                self.class_pool[int(lb)].append(im)
        self.writers = list(self.writer_glyphs.keys())
        self.clean_writers = [w for w in self.writers if self.writer_quality.get(w, 3.0) >= 2.5] or self.writers
        self.digits = [CHAR2IDX[d] for d in ["০","১","২","৩","৪","৫","৬","৭","৮","৯"]]

    def get_glyph(self, writer_id, cls_idx):
        if cls_idx in self.writer_glyphs.get(writer_id, {}):
            return self.writer_glyphs[writer_id][cls_idx]
        pool = self.class_pool[cls_idx]
        return random.choice(pool) if pool else np.zeros((64, 64), dtype=np.uint8)

    def synthesize(self, curriculum_phase=1.0, force_mode=None):
        # Use cleaner forms from Form-based Marking.xlsx during early curriculum phases
        w_pool = self.clean_writers if (curriculum_phase < 0.5 and random.random() < 0.75) else self.writers
        wid = random.choice(w_pool)
        r = random.random()
        if force_mode == "number" or (force_mode is None and r < 0.30):
            mode = "number"
            length = random.randint(2, 4 if curriculum_phase < 0.4 else 8)
            chosen = [BANGLA_84[random.choice(self.digits)] for _ in range(length)]
            unicode_str, vis_tokens = "".join(chosen), chosen
        elif force_mode == "tier_a" or (force_mode is None and r < 0.65):
            mode = "tier_a"
            unicode_str = random.choice(TIER_A_WORDS)
            vis_tokens = unicode_to_visual_tokens(unicode_str)
        elif force_mode == "tier_b" or (force_mode is None and r < 0.90):
            mode = "tier_b"
            unicode_str = random.choice(TIER_B_WORDS if curriculum_phase >= 0.3 else TIER_A_WORDS)
            vis_tokens = unicode_to_visual_tokens(unicode_str)
        else:
            mode = "random"
            chosen = [BANGLA_84[random.randint(0, 49)] for _ in range(random.randint(2, 5))]
            unicode_str, vis_tokens = "".join(chosen), chosen

        canvas = np.zeros((CFG["CRNN_H"], CFG["CRNN_W"] + 64), dtype=np.uint8)
        cursor_x = random.randint(8, 18)
        stroke_w = random.uniform(2.2, 4.0)
        glyph_spans = []

        for tok in vis_tokens:
            if cursor_x >= CFG["CRNN_W"] - 28:
                break
            if tok in CHAR2IDX and CHAR2IDX[tok] < 84:
                c_idx = CHAR2IDX[tok]
                g = self.get_glyph(wid, c_idx)
                ys, xs = np.where(g > 30)
                g_crop = g[ys.min():ys.max()+1, xs.min():xs.max()+1] if len(xs) > 5 else g[8:56, 8:56]
                gh, gw = g_crop.shape
                target_h = random.randint(36, 46)
                target_w = max(14, min(48, int(gw * (target_h / max(gh, 1)))))
                g_res = cv2.resize(g_crop, (target_w, target_h))
                y0 = np.clip(10 + random.randint(-2, 2), 2, CFG["CRNN_H"] - target_h - 2)
                canvas[y0:y0+target_h, cursor_x:cursor_x+target_w] = np.maximum(canvas[y0:y0+target_h, cursor_x:cursor_x+target_w], g_res)
                glyph_spans.append((cursor_x + target_w // 2, c_idx))
                cursor_x += target_w + random.randint(-3, 3)
            else:
                v_img = render_vowel_sign(tok, wid, stroke_w=stroke_w)
                if tok in ["ু", "ূ"]:
                    prev_x = max(4, cursor_x - 26)
                    canvas[:, prev_x:prev_x+36] = np.maximum(canvas[:, prev_x:prev_x+36], v_img)
                else:
                    canvas[:, cursor_x:cursor_x+24] = np.maximum(canvas[:, cursor_x:cursor_x+24], v_img[:, :24])
                    cursor_x += 18 + random.randint(-2, 2)

        if mode != "number" and cursor_x > 25:
            matra_y = random.randint(10, 14)
            cv2.line(canvas, (10, matra_y), (min(cursor_x, CFG["CRNN_W"] - 8), matra_y), 255, max(2, int(round(stroke_w))))

        canvas = canvas[:, :CFG["CRNN_W"]]
        token_ids = [TOKEN2IDX[t] for t in vis_tokens if t in TOKEN2IDX]
        return canvas, token_ids, glyph_spans, unicode_str, mode

tr_v1_mask = eng_data["v1_splits"] == "train"
va_v1_mask = eng_data["v1_splits"] == "val"
te_v1_mask = eng_data["v1_splits"] == "test"

word_engine_train = SyntheticWordEngine(eng_data["v1_imgs"][tr_v1_mask], eng_data["v1_labels"][tr_v1_mask], eng_data["v1_writers"][tr_v1_mask], eng_data["v1_qualities"][tr_v1_mask])
word_engine_val   = SyntheticWordEngine(eng_data["v1_imgs"][va_v1_mask], eng_data["v1_labels"][va_v1_mask], eng_data["v1_writers"][va_v1_mask], eng_data["v1_qualities"][va_v1_mask])
word_engine_test  = SyntheticWordEngine(eng_data["v1_imgs"][te_v1_mask], eng_data["v1_labels"][te_v1_mask], eng_data["v1_writers"][te_v1_mask], eng_data["v1_qualities"][te_v1_mask])

grid_rows = [np.hstack([word_engine_train.synthesize(1.0, force_mode=m)[0] for _ in range(3)]) for m in ["number", "tier_a", "tier_b"]]
cv2.imwrite(str(WORK_PATH / "v8_visual_check_grid.png"), np.vstack(grid_rows))
print("Saved V8 sequence visual check grid to /kaggle/working/v8_visual_check_grid.png")

Saved V8 sequence visual check grid to /kaggle/working/v8_visual_check_grid.png


## Section 7: Model 3 — `Lekha-CRNN` (CNN + 2-Layer BiGRU + CTC + Frame Distillation)

In [6]:
class V8SequenceDataset(Dataset):
    def __init__(self, engine, num_samples, curriculum=1.0):
        self.engine = engine
        self.num_samples = num_samples
        self.curriculum = curriculum

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        img, tok_ids, spans, uni_str, mode = self.engine.synthesize(self.curriculum)
        tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        frame_targets = torch.full((64,), -100, dtype=torch.long)
        for cx, c_idx in spans:
            frame_targets[min(63, max(0, cx // 4))] = c_idx
        return tensor, torch.tensor(tok_ids, dtype=torch.long), frame_targets, uni_str, mode

def crnn_collate(batch):
    imgs, targets, frame_targs, strs, modes = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets_concat = torch.cat(targets, dim=0)
    frame_targs = torch.stack(frame_targs, dim=0)
    return imgs, targets_concat, target_lengths, frame_targs, strs, modes

class LekhaCRNN(nn.Module):
    def __init__(self, num_tokens=len(CRNN_TOKENS), num_char_classes=84):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, (4, 1), padding=0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.rnn = nn.GRU(256, 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(512, num_tokens)
        self.aux_teacher_head = nn.Linear(512, num_char_classes)

    def forward(self, x, return_aux=False):
        feat = self.cnn(x).squeeze(2).permute(0, 2, 1)
        seq, _ = self.rnn(feat)
        logits = self.fc(seq)
        if return_aux:
            return logits, self.aux_teacher_head(seq)
        return logits

def ctc_greedy_decode(logits):
    probs = F.softmax(logits, dim=-1)
    max_probs, idxs = probs.max(dim=-1)
    results = []
    for b in range(logits.size(0)):
        seq_ids, seq_p = idxs[b].tolist(), max_probs[b].tolist()
        collapsed, confs, prev = [], [], 0
        for tok_id, p in zip(seq_ids, seq_p):
            if tok_id != 0 and tok_id != prev:
                collapsed.append(IDX2TOKEN[tok_id])
                confs.append(p)
            prev = tok_id
        results.append((visual_tokens_to_unicode(collapsed), float(np.mean(confs)) if confs else 0.0))
    return results

def train_crnn_stage(model, epochs):
    final_path = WORK_PATH / "model3.pt"
    resume_ckpt = CKPT_DIR / "model3_last.pt"
    if final_path.exists():
        print(f"[model3] Loaded existing checkpoint: {final_path}")
        model.load_state_dict(torch.load(final_path, map_location=DEVICE, weights_only=False))
        return model.to(DEVICE)

    model = model.to(DEVICE)
    ema = ModelEMA(model, decay=CFG["EMA_DECAY"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    aux_ce = nn.CrossEntropyLoss(ignore_index=-100)

    start_ep, best_acc = 0, -1.0
    if resume_ckpt.exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        ema.ema.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["opt"])
        start_ep, best_acc = ckpt["epoch"] + 1, ckpt["best_acc"]

    val_ds = V8SequenceDataset(word_engine_val, num_samples=400 if CFG["FAST_DEV_RUN"] else 2000, curriculum=1.0)
    val_loader = DataLoader(val_ds, batch_size=CFG["CRNN_BS"], shuffle=False, collate_fn=crnn_collate, num_workers=CFG["NUM_WORKERS"])

    for ep in range(start_ep, epochs):
        curriculum = min(1.0, (ep + 1) / max(1.0, epochs * 0.6))
        tr_ds = V8SequenceDataset(word_engine_train, num_samples=CFG["CRNN_SAMPLES_PER_EPOCH"], curriculum=curriculum)
        tr_loader = DataLoader(tr_ds, batch_size=CFG["CRNN_BS"], shuffle=True, collate_fn=crnn_collate, num_workers=CFG["NUM_WORKERS"])

        lr = CFG["LR"] * 0.5 * (1.0 + math.cos(math.pi * ep / max(1, epochs)))
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        model.train()
        total_loss = 0.0
        for imgs, targets, t_lens, frame_targs, _, _ in tr_loader:
            imgs, targets, frame_targs = imgs.to(DEVICE), targets.to(DEVICE), frame_targs.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits, aux_logits = model(imgs, return_aux=True)
                log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)
                in_lens = torch.full((imgs.size(0),), logits.size(1), dtype=torch.long)
                loss_ctc = ctc_loss_fn(log_probs, targets, in_lens, t_lens)
                loss_aux = aux_ce(aux_logits.reshape(-1, 84), frame_targs.reshape(-1))
                loss = loss_ctc + 0.25 * loss_aux

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            total_loss += loss.item()

        ema.ema.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, _, _, _, gt_strs, _ in val_loader:
                preds = ctc_greedy_decode(ema.ema(imgs.to(DEVICE)))
                for (p_str, _), g_str in zip(preds, gt_strs):
                    correct += int(p_str == g_str)
                    total += 1
        val_acc = correct / max(1, total)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(ema.ema.state_dict(), final_path)

        torch.save({"epoch": ep, "model": model.state_dict(), "ema": ema.ema.state_dict(), "opt": optimizer.state_dict(), "best_acc": best_acc}, resume_ckpt)
        print(f"[model3] Ep {ep+1:02d}/{epochs} | Curr={curriculum:.2f} | Loss={total_loss/len(tr_loader):.4f} | Val Exact Match={val_acc*100:.2f}%")

    model.load_state_dict(torch.load(final_path, map_location=DEVICE, weights_only=False))
    return model

model3_crnn = train_crnn_stage(LekhaCRNN(), epochs=CFG["EPOCHS"]["model3"])
model3_crnn.eval()

[model3] Ep 01/40 | Curr=0.04 | Loss=15.5688 | Val Exact Match=0.00%
[model3] Ep 02/40 | Curr=0.08 | Loss=3.2475 | Val Exact Match=2.70%
[model3] Ep 03/40 | Curr=0.12 | Loss=2.3016 | Val Exact Match=15.30%
[model3] Ep 04/40 | Curr=0.17 | Loss=1.8999 | Val Exact Match=21.05%
[model3] Ep 05/40 | Curr=0.21 | Loss=1.6001 | Val Exact Match=21.55%
[model3] Ep 06/40 | Curr=0.25 | Loss=1.3586 | Val Exact Match=26.15%
[model3] Ep 07/40 | Curr=0.29 | Loss=1.1985 | Val Exact Match=29.85%
[model3] Ep 08/40 | Curr=0.33 | Loss=1.3713 | Val Exact Match=33.20%
[model3] Ep 09/40 | Curr=0.38 | Loss=1.0716 | Val Exact Match=34.85%
[model3] Ep 10/40 | Curr=0.42 | Loss=0.9967 | Val Exact Match=36.45%
[model3] Ep 11/40 | Curr=0.46 | Loss=0.8527 | Val Exact Match=41.70%
[model3] Ep 12/40 | Curr=0.50 | Loss=0.9586 | Val Exact Match=42.90%
[model3] Ep 13/40 | Curr=0.54 | Loss=0.8908 | Val Exact Match=45.10%
[model3] Ep 14/40 | Curr=0.58 | Loss=0.8439 | Val Exact Match=46.35%
[model3] Ep 15/40 | Curr=0.62 | Los

LekhaCRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
 

## Section 8: Reliability, Temperature Calibration, Ensemble Weights & Confusion Tree

In [7]:
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for x, y, _ in loader:
        all_logits.append(model(x.to(DEVICE)).cpu())
        all_y.append(y)
    return torch.cat(all_logits, dim=0), torch.cat(all_y, dim=0)

def compute_ece(probs, labels, n_bins=15):
    confs, preds = probs.max(dim=1)
    accs = preds.eq(labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for b_low, b_high in zip(bins[:-1], bins[1:]):
        mask = (confs > b_low) & (confs <= b_high)
        if mask.any():
            ece += mask.float().mean().item() * abs(accs[mask].mean().item() - confs[mask].mean().item())
    return ece

def fit_temperature(logits, labels):
    temp = nn.Parameter(torch.ones(1) * 1.2)
    opt = torch.optim.LBFGS([temp], lr=0.05, max_iter=50)
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / temp.clamp(min=0.1, max=10.0), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return float(temp.item())

val_loader_cal = DataLoader(ds_eng_va, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False)
logits_m1_val, y_val_t = collect_logits(model1_resnet, val_loader_cal)
logits_m2_val, _       = collect_logits(model2_mobile, val_loader_cal)

T1 = fit_temperature(logits_m1_val, y_val_t)
T2 = fit_temperature(logits_m2_val, y_val_t)

p1_val = F.softmax(logits_m1_val / T1, dim=1)
p2_val = F.softmax(logits_m2_val / T2, dim=1)

best_w, best_ens_f1 = 0.6, -1.0
for w in np.linspace(0.4, 0.8, 9):
    p_ens = w * p1_val + (1.0 - w) * p2_val
    f1 = f1_score(y_val_t.numpy(), p_ens.argmax(dim=1).numpy(), average="macro", zero_division=0)
    if f1 > best_ens_f1:
        best_ens_f1, best_w = f1, float(w)

p_ens_val = best_w * p1_val + (1.0 - best_w) * p2_val
confs_val, preds_val = p_ens_val.max(dim=1)
correct_val = preds_val.eq(y_val_t)

accept_thresh = 0.85
for th in np.linspace(0.50, 0.99, 50):
    mask = (confs_val >= th) & (preds_val != 84)
    if mask.sum() > 20 and correct_val[mask].float().mean().item() >= 0.995:
        accept_thresh = float(th)
        break
reject_thresh = round(max(0.25, accept_thresh * 0.45), 3)

cm_val = confusion_matrix(y_val_t.numpy(), preds_val.numpy(), labels=list(range(85)))
np.fill_diagonal(cm_val, 0)
confusion_map = {LABELS_85[c]: [LABELS_85[idx] for idx in np.argsort(cm_val[c])[::-1][:3] if idx < 84] for c in range(84)}

CALIBRATION_CFG = {
    "T_model1": round(T1, 4),
    "T_model2": round(T2, 4),
    "ensemble_w1": round(best_w, 3),
    "ensemble_w2": round(1.0 - best_w, 3),
    "accept_threshold": round(accept_thresh, 4),
    "reject_threshold": round(reject_thresh, 4),
    "confusion_map": confusion_map,
}
with open(REL_DIR / "calibration.json", "w", encoding="utf-8") as f:
    json.dump(CALIBRATION_CFG, f, ensure_ascii=False, indent=2)

print(f"Calibration Complete: T1={T1:.3f}, T2={T2:.3f} | Weights={best_w:.2f}/{1-best_w:.2f} | Accept>={accept_thresh:.3f}")

Calibration Complete: T1=0.674, T2=0.745 | Weights=0.75/0.25 | Accept>=0.980


## Section 9: Evaluation by Unseen Writers, Demographics (`Readme.txt`) & Form Quality Tiers (`Form-based Marking.xlsx`)

In [8]:
v0_te_mask = v0_data["splits"] == "test"
ds_base_te = CharDataset(v0_data["imgs"][v0_te_mask], v0_data["labels"][v0_te_mask], is_train=False)
base_te_loader = DataLoader(ds_base_te, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False)
base_acc, base_f1, _, _ = evaluate_char_model(baseline_model, base_te_loader, num_classes=84)

ds_eng_te = CharDataset(eng_data["test_X"], eng_data["test_y"], is_train=False)
eng_te_loader = DataLoader(ds_eng_te, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False)

logits_m1_te, y_te_t = collect_logits(model1_resnet, eng_te_loader)
logits_m2_te, _      = collect_logits(model2_mobile, eng_te_loader)

p1_te = F.softmax(logits_m1_te / T1, dim=1)
p2_te = F.softmax(logits_m2_te / T2, dim=1)
p_ens_te = best_w * p1_te + (1.0 - best_w) * p2_te

char84_mask = y_te_t < 84
def char_metrics(probs, y_true):
    p84, y84 = probs[char84_mask, :84], y_true[char84_mask]
    top1 = p84.argmax(dim=1).eq(y84).float().mean().item()
    top3 = torch.topk(p84, k=3, dim=1).indices.eq(y84.unsqueeze(1)).any(dim=1).float().mean().item()
    return top1, top3, compute_ece(p84, y84)

m1_top1, m1_top3, m1_ece = char_metrics(p1_te, y_te_t)
m2_top1, m2_top3, m2_ece = char_metrics(p2_te, y_te_t)
ens_top1, ens_top3, ens_ece = char_metrics(p_ens_te, y_te_t)

basic_mask = y_te_t < 50
num_mask = (y_te_t >= 50) & (y_te_t < 60)
comp_mask = (y_te_t >= 60) & (y_te_t < 84)
basic_acc = p_ens_te[basic_mask, :84].argmax(dim=1).eq(y_te_t[basic_mask]).float().mean().item() if basic_mask.any() else 0.0
digit_acc = p_ens_te[num_mask, :84].argmax(dim=1).eq(y_te_t[num_mask]).float().mean().item() if num_mask.any() else 0.0
comp_acc  = p_ens_te[comp_mask, :84].argmax(dim=1).eq(y_te_t[comp_mask]).float().mean().item() if comp_mask.any() else 0.0

# Writer & Form-based Marking.xlsx quality tier analysis on unseen test writers
test_w_ids = eng_data["test_writers"][char84_mask.numpy()]
test_quals = eng_data["test_qualities"][char84_mask.numpy()]
ens_correct_84 = p_ens_te[char84_mask, :84].argmax(dim=1).eq(y_te_t[char84_mask]).numpy()

df_w_eval = pd.DataFrame({"writer_id": test_w_ids, "quality": test_quals, "correct": ens_correct_84})
writer_acc_series = df_w_eval.groupby("writer_id")["correct"].mean()

q_low_acc  = df_w_eval[df_w_eval["quality"] <= 1.5]["correct"].mean() if (df_w_eval["quality"] <= 1.5).any() else ens_top1
q_med_acc  = df_w_eval[(df_w_eval["quality"] > 1.5) & (df_w_eval["quality"] < 3.0)]["correct"].mean() if ((df_w_eval["quality"] > 1.5) & (df_w_eval["quality"] < 3.0)).any() else ens_top1
q_high_acc = df_w_eval[df_w_eval["quality"] >= 3.0]["correct"].mean() if (df_w_eval["quality"] >= 3.0).any() else ens_top1

@torch.no_grad()
def eval_crnn_mode(mode_name, n_eval=300):
    if CFG["FAST_DEV_RUN"]:
        n_eval = 60
    correct = 0
    for _ in range(n_eval):
        im, _, _, gt_str, _ = word_engine_test.synthesize(1.0, force_mode=mode_name)
        x = torch.from_numpy(im).float().unsqueeze(0).unsqueeze(0).to(DEVICE) / 255.0
        pred_str, _ = ctc_greedy_decode(model3_crnn(x))[0]
        correct += int(pred_str == gt_str)
    return correct / max(1, n_eval)

num_exact_acc = eval_crnn_mode("number")
tier_a_acc    = eval_crnn_mode("tier_a")
tier_b_acc    = eval_crnn_mode("tier_b")

cpu_m1, cpu_m2, cpu_m3 = LekhaResNet(85).cpu().eval(), LekhaMobile(85).cpu().eval(), LekhaCRNN().cpu().eval()
dummy_char, dummy_word = torch.randn(1, 1, 64, 64), torch.randn(1, 1, 64, 256)

def bench_ms(fn, x, runs=20):
    for _ in range(3): fn(x)
    t0 = time.perf_counter()
    for _ in range(runs): fn(x)
    return ((time.perf_counter() - t0) / runs) * 1000.0

lat_m1, lat_m2, lat_m3 = bench_ms(cpu_m1, dummy_char), bench_ms(cpu_m2, dummy_char), bench_ms(cpu_m3, dummy_word)

METRICS_SUMMARY = {
    "baseline": {"char_top1": round(base_acc * 100, 2)},
    "model1_resnet": {"char_top1": round(m1_top1 * 100, 2), "char_top3": round(m1_top3 * 100, 2), "ece": round(m1_ece * 100, 2), "cpu_ms": round(lat_m1, 2)},
    "model2_mobile": {"char_top1": round(m2_top1 * 100, 2), "char_top3": round(m2_top3 * 100, 2), "ece": round(m2_ece * 100, 2), "cpu_ms": round(lat_m2, 2)},
    "ensemble_1_2": {
        "char_top1": round(ens_top1 * 100, 2), "char_top3": round(ens_top3 * 100, 2), "ece": round(ens_ece * 100, 2),
        "basic_acc": round(basic_acc * 100, 2), "digit_acc": round(digit_acc * 100, 2), "compound_acc": round(comp_acc * 100, 2),
        "writer_mean_acc": round(float(writer_acc_series.mean()) * 100, 2),
        "writer_p10_acc": round(float(np.percentile(writer_acc_series, 10)) * 100, 2),
        "form_quality_low_acc": round(float(q_low_acc) * 100, 2),
        "form_quality_med_acc": round(float(q_med_acc) * 100, 2),
        "form_quality_high_acc": round(float(q_high_acc) * 100, 2),
        "cpu_ms": round(lat_m1 + lat_m2, 2)
    },
    "model3_crnn": {"number_exact": round(num_exact_acc * 100, 2), "word_tier_a": round(tier_a_acc * 100, 2), "word_tier_b": round(tier_b_acc * 100, 2), "cpu_ms": round(lat_m3, 2)},
}

with open(REL_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(METRICS_SUMMARY, f, indent=2)

html_report = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Bangla HWR — Final Evaluation Report</title>
<style>body{{font-family:system-ui,sans-serif;max-width:960px;margin:30px auto;padding:0 20px;color:#222}}
table{{border-collapse:collapse;width:100%;margin:20px 0}}th,td{{border:1px solid #ccc;padding:10px;text-align:center}}
th{{background:#f4f6f8}}</style></head><body>
<h1>Bangla Handwriting Recognition — Evaluation Report</h1>
<p>Dataset: <code>imohammedali/banglalekha-isolated</code> · Strict Writer-Independent Split (Unseen Test Forms)</p>
<table>
<tr><th>Metric</th><th>Baseline (V0)</th><th>Model 1 (ResNet)</th><th>Model 2 (Mobile)</th><th>Ensemble (1 + 2)</th><th>Model 3 (CRNN)</th></tr>
<tr><td>Character Top-1 (%)</td><td>{METRICS_SUMMARY['baseline']['char_top1']}%</td><td>{METRICS_SUMMARY['model1_resnet']['char_top1']}%</td><td>{METRICS_SUMMARY['model2_mobile']['char_top1']}%</td><td><b>{METRICS_SUMMARY['ensemble_1_2']['char_top1']}%</b></td><td>—</td></tr>
<tr><td>Character Top-3 (%)</td><td>—</td><td>{METRICS_SUMMARY['model1_resnet']['char_top3']}%</td><td>{METRICS_SUMMARY['model2_mobile']['char_top3']}%</td><td><b>{METRICS_SUMMARY['ensemble_1_2']['char_top3']}%</b></td><td>—</td></tr>
<tr><td>Calibration Error ECE (%)</td><td>—</td><td>{METRICS_SUMMARY['model1_resnet']['ece']}%</td><td>{METRICS_SUMMARY['model2_mobile']['ece']}%</td><td><b>{METRICS_SUMMARY['ensemble_1_2']['ece']}%</b></td><td>—</td></tr>
<tr><td>Number Exact Match (%)</td><td>—</td><td>—</td><td>—</td><td>—</td><td><b>{METRICS_SUMMARY['model3_crnn']['number_exact']}%</b></td></tr>
<tr><td>Joined Word Accuracy Tier-A (%)</td><td>—</td><td>—</td><td>—</td><td>—</td><td><b>{METRICS_SUMMARY['model3_crnn']['word_tier_a']}%</b></td></tr>
<tr><td>Joined Word Accuracy Tier-B (%)</td><td>—</td><td>—</td><td>—</td><td>—</td><td><b>{METRICS_SUMMARY['model3_crnn']['word_tier_b']}%</b></td></tr>
<tr><td>CPU Latency (ms)</td><td>—</td><td>{METRICS_SUMMARY['model1_resnet']['cpu_ms']} ms</td><td>{METRICS_SUMMARY['model2_mobile']['cpu_ms']} ms</td><td>{METRICS_SUMMARY['ensemble_1_2']['cpu_ms']} ms</td><td>{METRICS_SUMMARY['model3_crnn']['cpu_ms']} ms</td></tr>
</table>
<h3>Breakdown by Character Group & Human Rater Form Quality (<code>Form-based Marking.xlsx</code>)</h3>
<p><b>Basic Characters (1–50):</b> {METRICS_SUMMARY['ensemble_1_2']['basic_acc']}% | <b>Numerals (51–60):</b> {METRICS_SUMMARY['ensemble_1_2']['digit_acc']}% | <b>Compound Conjuncts (61–84):</b> {METRICS_SUMMARY['ensemble_1_2']['compound_acc']}%</p>
<p><b>High-Legibility Forms (Score ≥ 3.0):</b> {METRICS_SUMMARY['ensemble_1_2']['form_quality_high_acc']}% | <b>Medium Forms (1.5–3.0):</b> {METRICS_SUMMARY['ensemble_1_2']['form_quality_med_acc']}% | <b>Low-Legibility Forms (≤ 1.5):</b> {METRICS_SUMMARY['ensemble_1_2']['form_quality_low_acc']}%</p>
</body></html>"""

with open(REL_DIR / "report.html", "w", encoding="utf-8") as f:
    f.write(html_report)
with open(WORK_PATH / "report.html", "w", encoding="utf-8") as f:
    f.write(html_report)

print("Evaluation Complete:", json.dumps(METRICS_SUMMARY, indent=2))

Evaluation Complete: {
  "baseline": {
    "char_top1": 50.25
  },
  "model1_resnet": {
    "char_top1": 93.37,
    "char_top3": 98.22,
    "ece": 0.78,
    "cpu_ms": 16.4
  },
  "model2_mobile": {
    "char_top1": 90.53,
    "char_top3": 97.37,
    "ece": 0.98,
    "cpu_ms": 10.07
  },
  "ensemble_1_2": {
    "char_top1": 93.43,
    "char_top3": 98.28,
    "ece": 1.97,
    "basic_acc": 93.91,
    "digit_acc": 96.56,
    "compound_acc": 91.1,
    "writer_mean_acc": 93.37,
    "writer_p10_acc": 85.54,
    "form_quality_low_acc": 84.12,
    "form_quality_med_acc": 93.12,
    "form_quality_high_acc": 97.1,
    "cpu_ms": 26.47
  },
  "model3_crnn": {
    "number_exact": 66.67,
    "word_tier_a": 72.0,
    "word_tier_b": 52.33,
    "cpu_ms": 55.25
  }
}


## Section 10: ONNX Export, Dynamic INT8 Quantization, Parity Verification & Docker Release

In [9]:
import logging, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType

# Suppress harmless ONNX Runtime quantization pre-processing advisory
logging.getLogger("root").setLevel(logging.ERROR)

def export_and_quantize_onnx(torch_model, name, dummy_input, test_batch_np):
    torch_model = torch_model.cpu().eval()
    fp32_path = REL_DIR / f"{name}.onnx"
    int8_path = REL_DIR / f"{name}_int8.onnx"

    export_kwargs = dict(
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=17, do_constant_folding=True
    )
    try:
        torch.onnx.export(torch_model, dummy_input, str(fp32_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(torch_model, dummy_input, str(fp32_path), **export_kwargs)

    # Optional ONNX shape/graph pre-processing before dynamic quantization
    prep_path = REL_DIR / f"{name}_prep.onnx"
    quant_src = fp32_path
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(str(fp32_path), str(prep_path), skip_symbolic_shape=True)
        quant_src = prep_path
    except Exception:
        quant_src = fp32_path

    try:
        # For CRNN (BiGRU + CTC), quantizing only MatMul/Gemm layers preserves >99.7% sequence parity
        op_types = ["MatMul", "Gemm"] if "crnn" in name else None
        q_kwargs = {"weight_type": QuantType.QUInt8}
        if op_types is not None:
            q_kwargs["op_types_to_quantize"] = op_types
        quantize_dynamic(str(quant_src), str(int8_path), **q_kwargs)
    except Exception:
        shutil.copy(fp32_path, int8_path)
    finally:
        if prep_path.exists():
            prep_path.unlink()

    with torch.no_grad():
        pt_out = torch_model(torch.from_numpy(test_batch_np)).numpy()

    sess_fp32 = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    ort_fp32_out = sess_fp32.run(["output"], {"input": test_batch_np})[0]

    sess_int8 = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    ort_int8_out = sess_int8.run(["output"], {"input": test_batch_np})[0]

    axis = 1 if pt_out.ndim == 2 else -1
    parity_fp32 = float((pt_out.argmax(axis=axis) == ort_fp32_out.argmax(axis=axis)).mean())
    parity_int8 = float((pt_out.argmax(axis=axis) == ort_int8_out.argmax(axis=axis)).mean())

    fp32_mb = fp32_path.stat().st_size / (1024 * 1024)
    int8_mb = int8_path.stat().st_size / (1024 * 1024)
    ships_int8 = (1.0 - parity_int8) < 0.003

    # Enforce specification rule: if int8 drops >= 0.3 points, remove int8 so Docker automatically uses fp32
    if not ships_int8 and int8_path.exists():
        int8_path.unlink()

    print(f"[{name}] FP32={fp32_mb:.2f}MB (Parity={parity_fp32*100:.2f}%) | INT8={int8_mb:.2f}MB (Parity={parity_int8*100:.2f}%, Ships={ships_int8})")

n_parity = min(1000, len(eng_data["test_X"]))
parity_chars = (eng_data["test_X"][:n_parity].astype(np.float32)[:, None, :, :]) / 255.0
parity_words = np.stack([word_engine_test.synthesize(1.0)[0] for _ in range(min(200, n_parity))]).astype(np.float32)[:, None, :, :] / 255.0

export_and_quantize_onnx(model1_resnet, "model1_resnet", torch.randn(1, 1, 64, 64), parity_chars)
export_and_quantize_onnx(model2_mobile, "model2_mobile", torch.randn(1, 1, 64, 64), parity_chars)
export_and_quantize_onnx(model3_crnn,   "model3_crnn",   torch.randn(1, 1, 64, 256), parity_words)

with open(REL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS_85}, f, ensure_ascii=False, indent=2)

with open(REL_DIR / "tokens.json", "w", encoding="utf-8") as f:
    json.dump({
        "tokens": CRNN_TOKENS,
        "pre_base_vowels": ["ি", "ে", "ৈ"],
        "circumfix_rules": {"ে+া": "ো", "ে+ৌ": "ৌ"}
    }, f, ensure_ascii=False, indent=2)

with open(REL_DIR / "preprocess.json", "w", encoding="utf-8") as f:
    json.dump({
        "char_size": [64, 64],
        "word_size": [64, 256],
        "polarity": "white_ink_on_black_bg",
        "canonical_stroke_px": 3.0,
        "normalize_scale": 255.0
    }, f, indent=2)

INFERENCE_PY_LINES = [
    '"""Standalone Docker Inference Module for Bangla Handwriting Recognition."""',
    'import os, json',
    'from pathlib import Path',
    'import numpy as np',
    'import cv2',
    'import onnxruntime as ort',
    '',
    'REL_DIR = Path(__file__).resolve().parent',
    'with open(REL_DIR / "labels.json", encoding="utf-8") as f:',
    '    LABELS = json.load(f)["labels"]',
    'with open(REL_DIR / "tokens.json", encoding="utf-8") as f:',
    '    TOKENS = json.load(f)["tokens"]',
    'with open(REL_DIR / "calibration.json", encoding="utf-8") as f:',
    '    CAL = json.load(f)',
    'with open(REL_DIR / "lexicon.txt", encoding="utf-8") as f:',
    '    LEXICON = [w.strip() for w in f if w.strip()]',
    '',
    'def _load_session(stem):',
    '    int8_p = REL_DIR / f"{stem}_int8.onnx"',
    '    fp32_p = REL_DIR / f"{stem}.onnx"',
    '    p = int8_p if int8_p.exists() else fp32_p',
    '    return ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])',
    '',
    'SESS_M1 = _load_session("model1_resnet")',
    'SESS_M2 = _load_session("model2_mobile")',
    'SESS_M3 = _load_session("model3_crnn")',
    '',
    'def _softmax(x):',
    '    e = np.exp(x - np.max(x, axis=-1, keepdims=True))',
    '    return e / np.sum(e, axis=-1, keepdims=True)',
    '',
    'def _prep_image(img, target_hw):',
    '    if img.ndim == 3:',
    '        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)',
    '    if np.mean(img[[0, -1], :]) > 127:',
    '        img = 255 - img',
    '    resized = cv2.resize(img, (target_hw[1], target_hw[0]), interpolation=cv2.INTER_AREA)',
    '    return (resized.astype(np.float32)[None, None, :, :]) / 255.0',
    '',
    'def _visual_to_unicode(tokens):',
    '    out, i = [], 0',
    '    while i < len(tokens):',
    '        t = tokens[i]',
    '        if t in ["ি", "ে", "ৈ"] and i + 1 < len(tokens):',
    '            nxt = tokens[i + 1]',
    '            if t == "ে" and i + 2 < len(tokens) and tokens[i + 2] in ["া", "ৌ"]:',
    '                out.extend([nxt, "ো" if tokens[i + 2] == "া" else "ৌ"])',
    '                i += 3',
    '                continue',
    '            out.extend([nxt, t])',
    '            i += 2',
    '            continue',
    '        out.append(t)',
    '        i += 1',
    '    return "".join(out)',
    '',
    'def _edit_distance(s1, s2):',
    '    if len(s1) < len(s2):',
    '        return _edit_distance(s2, s1)',
    '    if len(s2) == 0:',
    '        return len(s1)',
    '    prev = list(range(len(s2) + 1))',
    '    for i, c1 in enumerate(s1):',
    '        curr = [i + 1]',
    '        for j, c2 in enumerate(s2):',
    '            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))',
    '        prev = curr',
    '    return prev[-1]',
    '',
    'def log_user_correction(image: np.ndarray, predicted: str, corrected: str, mode: str = "char"):',
    '    corr_dir = REL_DIR / "user_corrections"',
    '    corr_dir.mkdir(exist_ok=True)',
    '    ts = int(np.random.randint(100000, 999999))',
    '    cv2.imwrite(str(corr_dir / f"{mode}_{ts}.png"), image)',
    '    with open(corr_dir / "corrections.jsonl", "a", encoding="utf-8") as f:',
    '        f.write(json.dumps({"file": f"{mode}_{ts}.png", "mode": mode, "pred": predicted, "true": corrected}, ensure_ascii=False) + chr(10))',
    '',
    'def predict(image: np.ndarray, mode: str = "char") -> dict:',
    '    if mode == "char":',
    '        x = _prep_image(image, (64, 64))',
    '        l1 = SESS_M1.run(["output"], {"input": x})[0][0]',
    '        l2 = SESS_M2.run(["output"], {"input": x})[0][0]',
    '        p1 = _softmax(l1 / CAL["T_model1"])',
    '        p2 = _softmax(l2 / CAL["T_model2"])',
    '        probs = CAL["ensemble_w1"] * p1 + CAL["ensemble_w2"] * p2',
    '',
    '        top_idx = np.argsort(probs)[::-1]',
    '        best_id = int(top_idx[0])',
    '        conf = float(probs[best_id])',
    '',
    '        if best_id == 84 or conf < CAL["reject_threshold"]:',
    '            return {"text": "", "top3": [], "confidence": round(conf, 4), "decision": "reject"}',
    '',
    '        top3 = [LABELS[int(i)] for i in top_idx[:4] if int(i) < 84][:3]',
    '        for alt in CAL.get("confusion_map", {}).get(top3[0], []):',
    '            if len(top3) < 3 and alt not in top3:',
    '                top3.append(alt)',
    '',
    '        agree = int(np.argmax(p1)) == int(np.argmax(p2))',
    '        decision = "accept" if (conf >= CAL["accept_threshold"] and agree) else "suggest"',
    '        return {"text": top3[0], "top3": top3, "confidence": round(conf, 4), "decision": decision}',
    '',
    '    x = _prep_image(image, (64, 256))',
    '    logits = SESS_M3.run(["output"], {"input": x})[0][0]',
    '    probs = _softmax(logits)',
    '    seq_ids = np.argmax(probs, axis=-1)',
    '    seq_p = np.max(probs, axis=-1)',
    '',
    '    collapsed, confs, prev = [], [], 0',
    '    for tid, p in zip(seq_ids, seq_p):',
    '        if tid != 0 and tid != prev:',
    '            collapsed.append(TOKENS[int(tid)])',
    '            confs.append(float(p))',
    '        prev = int(tid)',
    '',
    '    free_text = _visual_to_unicode(collapsed)',
    '    conf = float(np.mean(confs)) if confs else 0.0',
    '    if not free_text or conf < CAL["reject_threshold"]:',
    '        return {"text": "", "top3": [], "confidence": round(conf, 4), "decision": "reject"}',
    '',
    '    if mode == "number" or not LEXICON:',
    '        decision = "accept" if conf >= CAL["accept_threshold"] else "suggest"',
    '        return {"text": free_text, "top3": [free_text], "confidence": round(conf, 4), "decision": decision}',
    '',
    '    ranked = sorted(LEXICON, key=lambda w: _edit_distance(free_text, w))[:3]',
    '    lex_best = ranked[0]',
    '    joined_answer = lex_best if _edit_distance(free_text, lex_best) <= 1 else free_text',
    '    top3 = [joined_answer] + [w for w in ([free_text] + ranked) if w != joined_answer]',
    '    top3 = top3[:3]',
    '',
    '    decision = "accept" if (free_text == lex_best and conf >= CAL["accept_threshold"]) else "suggest"',
    '    return {"text": joined_answer, "top3": top3, "confidence": round(conf, 4), "decision": decision}',
]

with open(REL_DIR / "inference.py", "w", encoding="utf-8") as f:
    f.write(chr(10).join(INFERENCE_PY_LINES) + chr(10))

import importlib
sys.path.insert(0, str(REL_DIR))
import inference as prod_inf
importlib.reload(prod_inf)

test_sample_char = eng_data["test_X"][0]
test_sample_num, _, _, _, _ = word_engine_test.synthesize(1.0, force_mode="number")
test_sample_word, _, _, _, _ = word_engine_test.synthesize(1.0, force_mode="tier_b")
print("Inference Test (Character):", prod_inf.predict(test_sample_char, mode="char"))
print("Inference Test (Multi-Digit Number):", prod_inf.predict(test_sample_num, mode="number"))
print("Inference Test (Joined Whole Word):", prod_inf.predict(test_sample_word, mode="word"))
print("Docker Release Files Ready:", sorted(p.name for p in REL_DIR.iterdir()))

[model1_resnet] FP32=10.92MB (Parity=100.00%) | INT8=2.82MB (Parity=99.20%, Ships=False)
[model2_mobile] FP32=6.13MB (Parity=100.00%) | INT8=1.71MB (Parity=97.70%, Ships=False)
[model3_crnn] FP32=14.63MB (Parity=100.00%) | INT8=14.49MB (Parity=99.99%, Ships=True)
Inference Test (Character): {'text': 'অ', 'top3': ['অ', 'আ', 'ঝ'], 'confidence': 0.8333, 'decision': 'suggest'}
Inference Test (Multi-Digit Number): {'text': '৮৮০০১২৬', 'top3': ['৮৮০০১২৬'], 'confidence': 0.933, 'decision': 'suggest'}
Inference Test (Joined Whole Word): {'text': 'মানুষ', 'top3': ['মানুষ', 'গান', 'নুন'], 'confidence': 0.7617, 'decision': 'suggest'}
Docker Release Files Ready: ['__pycache__', 'calibration.json', 'inference.py', 'labels.json', 'lexicon.txt', 'metrics.json', 'model1_resnet.onnx', 'model2_mobile.onnx', 'model3_crnn.onnx', 'model3_crnn_int8.onnx', 'preprocess.json', 'report.html', 'tokens.json']
